In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

def load_ecg_data(filepath):
    df = pd.read_excel(filepath)
    
    def parse_signal(x):
        if not isinstance(x, str):
            return np.array([])

        values = []
        for i in x.split():
            try:
                values.append(float(i))
            except ValueError:
                continue

        return np.array(values)

    df["ecg_signal"] = df["ecg_400_str"].apply(parse_signal)

    df["fs"] = df["fs_out"]

    print("Dataset Loaded:", df.shape)

    return df

def bandpass_filter(signal, fs, low=0.5, high=40, order=3):

    signal = np.array(signal)

    #Check minimum length so it can run with filtfilt.
    if len(signal) < 30:
        return signal
    
    #Get Nyquist Frequency and convert to normalized values (0-1).
    nyquist = 0.5 * fs
    low = low / nyquist
    high = high / nyquist

    #Use Butterworth filter (frequency flat as possible).
    b, a = butter(order, [low, high], btype="band")

    #Filtfilt filters forward and backwards, helps with phase distortion and time shift.
    try:
        filtered = filtfilt(b, a, signal)
    except ValueError:
        filtered = signal

    return filtered


def quality_measure(signal):

    #Make a dictionary.
    metrics = {}

    #If there is low variance, it is flat.
    metrics["variance"] = np.var(signal)

    #Measure the signal strength to make sure not too weak or an artifact.
    metrics["amplitude_range"] = np.max(signal) - np.min(signal)

    #Measure the hgih frequency noise using signal differences.
    metrics["noise_level"] = np.std(np.diff(signal))

    #Detect Constant segments, which would be flatlines.
    metrics["flatline_ratio"] = np.sum(np.abs(np.diff(signal)) < 1e-5) / len(signal)

    #Detect clipped signal, which could indicate faulty recording.
    metrics["saturation_ratio"] = (
        np.sum(signal == np.max(signal)) + np.sum(signal == np.min(signal))
    ) / len(signal)

    return metrics


def compute_variance_thresholds(df):
    #Get thresholds.
    variances = df["ecg_signal"].apply(np.var)
    variance_threshold = variances.quantile(0.1)

    return variance_threshold


def compute_noise_thresholds(df):

    noise_values = df["ecg_signal"].apply(
        lambda x: np.std(np.diff(x))
    )
    noise_threshold = noise_values.quantile(0.90)

    return noise_threshold


def quality_score(metrics, variance_threshold, noise_threshold):

    score = 0

    if metrics["variance"] > variance_threshold:
        score += 1

    if metrics["flatline_ratio"] < 0.1:
        score += 1

    if metrics["saturation_ratio"] < 0.02:
        score += 1

    if metrics["noise_level"] < noise_threshold:
        score += 1

    return score


def clean_ecg_data(df):

    variance_threshold = compute_variance_thresholds(df)
    noise_threshold = compute_noise_thresholds(df)

    def process(row):
        signal = row["ecg_signal"]
        fs = row["fs"]

        filtered = bandpass_filter(signal, fs)
        metrics = quality_measure(filtered)
        score = quality_score(metrics, variance_threshold, noise_threshold)

        return pd.Series({
            "quality_score": score,
            "ecg_filtered": filtered
        })

    #Apply processing to each row.
    df[["quality_score", "ecg_filtered"]] = df.apply(process, axis=1)

    df_clean = df[df["quality_score"] >= 4].copy()

    print("Low-Quality Segments Removed:", (df["quality_score"] < 4).sum())
    print("Remaining Segments:", len(df_clean))

    return df_clean

def filter_patients(df):

    #Only keep patients with enough good segments.
    patient_counts = df["pseudonym"].value_counts()
    valid_patients = patient_counts[patient_counts >= 2].index
    removed_patients = patient_counts[patient_counts < 2].index

    print("Patients removed:", list(removed_patients))

    df = df[df["pseudonym"].isin(valid_patients)].copy()

    return df


def save_clean_data(df, filename="clean_ecg_dataset.xlsx"):

    df.to_excel(filename, index=False)

    print("Clean dataset saved:", filename)


def plot_ecg_by_patient(df):
    output_dir = "ecg_cleaned_plots"
    os.makedirs(output_dir, exist_ok=True)

    for pseudonym, group in df.groupby("pseudonym"):
        
        n = len(group)
        fig, axes = plt.subplots(n, 1, figsize=(12, 3*n), sharex=True)
        
        if n == 1:
            axes = [axes]
        
        for ax, (_, row) in zip(axes, group.iterrows()):
            signal = row["ecg_filtered"]
            fs = row["fs"]
            
            if len(signal) == 0:
                continue

            t = np.arange(len(signal)) / fs
            
            ax.plot(t, signal)
            ax.set_ylabel("ECG (nu)")
            ax.grid(True)

        axes[-1].set_xlabel("Time (seconds)")
        fig.suptitle(f"Pseudonym: {pseudonym}", fontsize=14)
        
        plt.tight_layout()
        
        safe_name = str(pseudonym).replace(" ", "_")
        filepath = os.path.join(output_dir, f"{safe_name}.png")
        
        plt.savefig(filepath, dpi=300)
        plt.close(fig)


def run_pipeline(filepath):

    df = load_ecg_data(filepath)

    df = clean_ecg_data(df)

    df = filter_patients(df)

    save_clean_data(df)

    plot_ecg_by_patient(df)


run_pipeline("ECG.xlsx")

Dataset Loaded: (601, 13)
Low-Quality Segments Removed: 107
Remaining Segments: 494
Patients removed: ['rtu624', 'ofc899']
Clean dataset saved: clean_ecg_dataset.xlsx
